#table of raw data

In [ ]:
!pip install torch-geometric

In [ ]:
# Cell 2: Imports and Configuration
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.data import Data, Batch
import os
import pandas as pd
from tqdm.auto import tqdm
import random
import time

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Configuration
# Get the directory where this notebook is located
import pathlib
NOTEBOOK_DIR = pathlib.Path().absolute()
BASE_DIR = os.path.join(str(NOTEBOOK_DIR), 'Scale_Results')
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Results will be saved to: {BASE_DIR}")

# Resolution reduction factor (K=4 means 4x smaller in each dimension)
K = 4

# Original dimensions divided by K
ORIGINAL_X, ORIGINAL_Y, ORIGINAL_Z = 681 // K, 344 // K, max(12 // K, 3)

SCALES = [
    ('original', 1, 1, 1), ('2x1y1z', 2, 1, 1), ('1x2y1z', 1, 2, 1),
    ('1x1y2z', 1, 1, 2), ('2x2y1z', 2, 2, 1), ('1x2y2z', 1, 2, 2),
    ('2x2y2z', 2, 2, 2), ('3x1y1z', 3, 1, 1), ('3x2y1z', 3, 2, 1),
    ('3x1y2z', 3, 1, 2), ('3x2y2z', 3, 2, 2)
]

TEXTURES = ['sinusoid', 'colonies', 'linear', 'olympic', 'oval']

# Adjusted parameters for lower resolution
Z_FIXED = max(5 // K, 1)
MAX_RADIUS = max(10 // K, 3)
STEP_SIZE = max(5 // K, 2)

# Match tolerance: consider coordinates within this distance as a match
# Updated to match F1 file method (same as MAX_RADIUS or at least 4)
MATCH_TOLERANCE = max(MAX_RADIUS, 4)  # voxels

# ============================================================
# EUCLIDEAN DISTANCE PARAMETERS
# ============================================================
# Z resolution is different from X,Y resolution
# Z_SCALE_FACTOR: multiply z difference by this factor before computing Euclidean distance
# If Z resolution is half of X,Y resolution, set Z_SCALE_FACTOR = 2
# This converts z voxels to the same physical scale as x,y voxels
Z_SCALE_FACTOR = 2.0  # Z resolution is half of X,Y, so multiply z by 2

# All Euclidean distances are divided by this value for normalization
EUCLIDEAN_STEP = 1  # voxels - change this to scale the distances

print(f"\nResolution reduction factor: K={K}")
print(f"Match tolerance: {MATCH_TOLERANCE} voxels")
print(f"Z Scale Factor: {Z_SCALE_FACTOR} (z voxels multiplied by this in distance calculation)")
print(f"Euclidean step (normalization): {EUCLIDEAN_STEP} (all distances divided by this)")
print(f"Effective dimensions: {ORIGINAL_X}x{ORIGINAL_Y}x{ORIGINAL_Z}")
print(f"Textures: {TEXTURES}")
print(f"Scales: {len(SCALES)}")
print(f"Total experiments: {len(TEXTURES) * len(SCALES)}")

In [ ]:
# Cell 3: Texture Generation Functions (Vectorized)

def generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Sinusoidal wave patterns - vectorized for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    # Scale parameters based on dimensions
    y_center = y_dim // 3
    strip_size = max(y_dim // 15, 3)

    # Scale sine parameters proportionally
    sine_params = [
        (0, y_dim // 3, x_dim),      # amplitude, period scaled
        (1, y_dim // 6, x_dim // 2),
        (2, y_dim // 2, x_dim * 3 // 4)
    ]

    x_range = np.arange(x_dim)
    for channel, amplitude, period in sine_params:
        if period == 0:
            period = 1
        y_sine = y_center + amplitude * np.sin(2 * np.pi * x_range / period)
        for z in range(z_dim):
            for x in range(x_dim):
                y_c = int(round(y_sine[x]))
                y_start, y_end = max(0, y_c - strip_size//2), min(y_dim, y_c + strip_size//2 + 1)
                for y in range(y_start, y_end):
                    data[channel, np.random.randint(0, 11), z, y, x] = 1

    # Channel 3: horizontal line
    y_start, y_end = max(0, y_center - strip_size//2), min(y_dim, y_center + strip_size//2 + 1)
    for z in range(z_dim):
        for x in range(x_dim):
            for y in range(y_start, y_end):
                data[3, np.random.randint(0, 11), z, y, x] = 1
    return data


def generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Linear strip patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

    # Scale parameters for low resolution
    base_offset = x_dim // 18
    base_width = max(x_dim // 140, 2)

    for i in range(3):
        for z in range(z_dim):
            x_start = max(0, int((base_offset + base_offset*i) * x_scale))
            x_end = min(x_dim, int((base_offset + base_width + base_offset*i) * x_scale))
            for y in range(y_dim):
                for x in range(x_start, x_end):
                    data[0, np.random.randint(6, 11), z, y, x] = 1

            y_base = y_dim // 20
            y_start = max(0, int((y_base + y_base*i) * y_scale))
            y_end = min(y_dim, int((y_base + base_width + y_base*i) * y_scale))
            for y in range(y_start, y_end):
                for x in range(x_dim):
                    data[1, np.random.randint(6, 11), z, y, x] = 1

            strip_w = max(int(2 * max(x_scale, y_scale)), 1)
            diag_offset = x_dim // 12
            for c in [int(-diag_offset * x_scale), 0]:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y - x - c) <= strip_w:
                            data[2, np.random.randint(6, 11), z, y, x] = 1
            for d in [int(diag_offset * y_scale), int(2 * diag_offset * y_scale)]:
                for y in range(y_dim):
                    for x in range(x_dim):
                        if abs(y + x - d) <= strip_w:
                            data[3, np.random.randint(6, 11), z, y, x] = 1
    return data


def generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Olympic rings pattern - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

    # Scale circles based on dimensions
    circles = [
        (0, x_dim * 3 // 10, y_dim * 4 // 10, min(x_dim, y_dim) // 4),
        (1, x_dim * 6 // 10, y_dim * 4 // 10, min(x_dim, y_dim) // 4),
        (2, x_dim * 45 // 100, y_dim * 7 // 10, min(x_dim, y_dim) // 4),
        (3, x_dim * 5 // 10, y_dim * 9 // 10, min(x_dim, y_dim) // 3)
    ]
    stripe_width = max(min(x_dim, y_dim) // 20, 2)

    for channel, cx, cy, radius in circles:
        inner_r, outer_r = max(radius - stripe_width, 1), radius
        for z in range(z_dim):
            for y in range(y_dim):
                for x in range(x_dim):
                    dist = np.sqrt((x-cx)**2 + (y-cy)**2)
                    if inner_r <= dist <= outer_r:
                        data[channel, np.random.randint(0, 11), z, y, x] = 1
    return data


def generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim):
    """Oval/ellipse patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    cx, cy = x_dim // 2, y_dim // 2
    x_stretch = 1.5

    # Scale radii based on dimensions
    min_dim = min(x_dim, y_dim)
    inner_r1, outer_r1 = min_dim // 4, min_dim // 2
    inner_r2, outer_r2 = min_dim * 4 // 10, min_dim * 55 // 100

    for z in range(z_dim):
        for y in range(y_dim):
            for x in range(x_dim):
                dx, dy = (x - cx) / x_stretch, y - cy
                dist = np.sqrt(dx**2 + dy**2)
                if inner_r1 <= dist <= outer_r1:
                    data[0, np.random.randint(6, 11), z, y, x] = 1
                if inner_r2 <= dist <= outer_r2:
                    data[1, np.random.randint(6, 11), z, y, x] = 1

    z_center = z_dim // 2
    cloud_r = (inner_r1 + outer_r1) // 2
    cloud_spread = max(min_dim // 10, 2)
    z_spread = max(z_dim // 4, 1)

    for _ in range(5):
        angle = np.random.uniform(0, 2*np.pi)
        r = np.random.uniform(cloud_r * 0.9, cloud_r * 1.1)
        cloud_cx = int(np.clip(cx + r * x_stretch * np.cos(angle), cloud_spread, x_dim - cloud_spread - 1))
        cloud_cy = int(np.clip(cy + r * np.sin(angle), cloud_spread, y_dim - cloud_spread - 1))
        for _ in range(max(20, min_dim // 5)):
            rx = np.random.randint(-cloud_spread, cloud_spread + 1)
            ry = np.random.randint(-cloud_spread, cloud_spread + 1)
            rz = np.random.randint(-z_spread, z_spread + 1)
            px, py, pz = cloud_cx + rx, cloud_cy + ry, z_center + rz
            if 0 <= px < x_dim and 0 <= py < y_dim and 0 <= pz < z_dim:
                data[2, np.random.randint(6, 11), pz, py, px] = 1
                data[3, np.random.randint(6, 11), pz, py, px] = 1
    return data


def generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, x_scale, y_scale):
    """Colony cloud patterns - scaled for low resolution"""
    data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
    min_dim = min(x_dim, y_dim)
    max_radius = max(min_dim // 8, 3)
    cloud_points = max(10, min_dim // 10)

    def add_cloud(channel, cx, cy, z, radius):
        for _ in range(cloud_points):
            angle = np.random.uniform(0, 2*np.pi)
            r = np.random.uniform(0, radius)
            px = int(np.clip(cx + r*np.cos(angle), 0, x_dim-1))
            py = int(np.clip(cy + r*np.sin(angle), 0, y_dim-1))
            data[channel, np.random.randint(6, 11), z, py, px] = 1

    # Scale step sizes
    arc_steps = max(x_dim // 8, 6)
    sine_step = max(x_dim // 8, 4)
    diag_step = max(x_dim // 6, 5)

    for z in range(z_dim):
        for t in np.linspace(0, 1, arc_steps):
            arc_x = int(t * (x_dim - 1))
            arc_y = int(t * (y_dim - 1) + 0.3 * (y_dim - 1) * np.sin(t * np.pi))
            arc_y = int(np.clip(arc_y, 0, y_dim-1))
            add_cloud(0, arc_x, arc_y, z, np.random.uniform(max_radius // 3, max_radius))

        for x in range(0, x_dim, sine_step):
            y_center = y_dim // 3
            y_amp = y_dim // 6
            y = int(y_center + y_amp * np.sin(2*np.pi*x/x_dim))
            y = int(np.clip(y, 0, y_dim-1))
            add_cloud(1, x, y, z, np.random.uniform(max_radius // 3, max_radius))

        diag_offsets = [y_dim // 5, y_dim * 2 // 5]
        for d in diag_offsets:
            for x in range(0, x_dim, diag_step):
                y = -x + int(d * y_scale)
                if 0 <= y < y_dim:
                    add_cloud(3, x, y, z, np.random.uniform(max_radius // 3, max_radius))
    return data


def create_groundtruth(texture_type, scale_name, x_scale, y_scale, z_scale, output_dir):
    """Create ground truth with specified texture and scale"""
    num_channels, num_values = 4, 11
    x_dim = ORIGINAL_X * x_scale
    y_dim = ORIGINAL_Y * y_scale
    z_dim = ORIGINAL_Z * z_scale
    local_x_scale, local_y_scale = x_dim / 172, y_dim / 87

    if texture_type == 'sinusoid':
        data = generate_sinusoid_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'linear':
        data = generate_linear_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    elif texture_type == 'olympic':
        data = generate_olympic_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'oval':
        data = generate_oval_texture(num_channels, num_values, z_dim, y_dim, x_dim)
    elif texture_type == 'colonies':
        data = generate_colonies_texture(num_channels, num_values, z_dim, y_dim, x_dim, local_x_scale, local_y_scale)
    else:
        raise ValueError(f"Unknown texture: {texture_type}")

    filepath = os.path.join(output_dir, f'groundtruth_{texture_type}_{scale_name}.npy')
    np.save(filepath, data)
    return data, filepath

print("Texture generation functions loaded!")

In [ ]:
# Cell 4: Subgraph Creation (Optimized)

def create_subgraphs(data, texture_type, scale_name, output_dir):
    """Fast subgraph creation using vectorized operations"""
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape
    z_idx = min(Z_FIXED, z_dim - 1)

    # Pre-compute mask
    intensity_matrix = np.zeros((y_dim, x_dim, num_channels), dtype=np.float32)
    channel_mask = np.zeros((y_dim, x_dim, num_channels), dtype=bool)

    for ch in range(num_channels):
        ch_data = data[ch, :, z_idx, :, :]
        mask = ch_data.sum(axis=0) > 0
        intensity_matrix[:, :, ch] = np.where(mask, np.argmax(ch_data, axis=0), 0)
        channel_mask[:, :, ch] = mask

    channel_counts = channel_mask.sum(axis=2)
    centers = [(x, y, z_idx) for x in range(0, x_dim, STEP_SIZE) for y in range(0, y_dim, STEP_SIZE)]

    all_subgraphs = []
    for cx, cy, cz in centers:
        x_min, x_max = max(0, cx - MAX_RADIUS), min(x_dim, cx + MAX_RADIUS + 1)
        y_min, y_max = max(0, cy - MAX_RADIUS), min(y_dim, cy + MAX_RADIUS + 1)

        nodes, positions, active_chs = [], [], []
        for y in range(y_min, y_max):
            for x in range(x_min, x_max):
                if channel_counts[y, x] > 0:
                    dist = np.sqrt((x-cx)**2 + (y-cy)**2)
                    if dist <= MAX_RADIUS:
                        active = np.where(channel_mask[y, x, :])[0].tolist()
                        nodes.append(intensity_matrix[y, x, active])
                        positions.append((x, y, z_idx))
                        active_chs.append(active)

        if len(nodes) < 2:
            continue

        max_ch = max(len(ch) for ch in active_chs)
        padded = []
        for i, n in enumerate(nodes):
            p = np.zeros(max_ch, dtype=np.float32)
            p[:len(n)] = n
            padded.append(p)

        node_features = np.array(padded, dtype=np.float32)
        pos_array = np.array(positions, dtype=np.int32)

        # Fast edge creation
        # Apply Z_SCALE_FACTOR to account for different z resolution (z is half resolution of x,y)
        diff = pos_array[:, np.newaxis, :] - pos_array[np.newaxis, :, :]
        diff[:, :, 2] = diff[:, :, 2] * Z_SCALE_FACTOR  # Scale z dimension
        dist_matrix = np.sqrt(np.sum(diff**2, axis=2))
        edge_mask = (dist_matrix <= MAX_RADIUS) & (dist_matrix > 0)
        edge_i, edge_j = np.where(edge_mask)

        if len(edge_i) == 0:
            continue

        edge_weights = dist_matrix[edge_i, edge_j].astype(np.float32)

        graph = Data(
            x=torch.tensor(node_features, dtype=torch.float32),
            edge_index=torch.tensor([edge_i, edge_j], dtype=torch.long),
            edge_attr=torch.tensor(edge_weights, dtype=torch.float32),
            center=(cx, cy, cz)
        )
        all_subgraphs.append(graph)

    return all_subgraphs

print("Subgraph creation function loaded!")

In [ ]:
# Cell 5: Model Definition (GPU Optimized)

class ContrastiveGAT(nn.Module):
    def __init__(self, in_channels=4, hidden=32, proj_dim=16, heads=4, dropout=0.1, edge_dim=None):
        super().__init__()
        self.edge_dim = edge_dim
        kw = dict(dropout=dropout)
        if edge_dim: kw['edge_dim'] = edge_dim

        self.gat1 = GATConv(in_channels, hidden, heads=heads, concat=True, **kw)
        self.gat2 = GATConv(hidden*heads, hidden, heads=heads, concat=True, **kw)
        self.gat3 = GATConv(hidden*heads, hidden, heads=1, concat=False, **kw)

        self.norm1 = nn.LayerNorm(hidden*heads)
        self.norm2 = nn.LayerNorm(hidden*heads)
        self.dropout = nn.Dropout(dropout)

        self.projection = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, proj_dim)
        )
        self.interaction_head = nn.Sequential(
            nn.Linear(hidden*3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden//2, 1)
        )

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        ea = edge_attr if self.edge_dim and edge_attr is not None else None
        x = F.elu(self.dropout(self.norm1(self.gat1(x, edge_index, ea))))
        x = F.elu(self.dropout(self.norm2(self.gat2(x, edge_index, ea))))
        x = F.elu(self.gat3(x, edge_index, ea))

        if batch is None:
            batch = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)

        emb = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch), global_add_pool(x, batch)], dim=1)
        proj = F.normalize(self.projection(emb), dim=1)
        score = self.interaction_head(emb)
        return proj, score


def prepare_graph(graph, target_channels=4):
    x = graph.x.clone()
    if x.shape[1] < target_channels:
        x = torch.cat([x, torch.zeros(x.shape[0], target_channels - x.shape[1])], dim=1)
    elif x.shape[1] > target_channels:
        x = x[:, :target_channels]
    g = Data(x=x, edge_index=graph.edge_index.clone())
    if hasattr(graph, 'edge_attr') and graph.edge_attr is not None:
        g.edge_attr = graph.edge_attr.clone()
    return g


def graph_augment(graph, target_channels=4):
    g = prepare_graph(graph, target_channels)
    num_nodes = g.x.shape[0]
    mask_n = int(num_nodes * 0.1)
    if mask_n > 0:
        g.x[torch.randperm(num_nodes)[:mask_n]] = 0.0
    return g


def contrastive_loss(z1, z2, temp=0.1):
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    sim = torch.matmul(z1, z2.T) / temp
    labels = torch.arange(z1.shape[0], device=z1.device)
    return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2


def train_model_gpu(model, graphs, device, epochs=1, batch_size=32, lr=0.01, gradient_accumulation_steps=4):
    """GPU optimized training with ranking loss for better top-k prediction"""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    target_ch = max(g.x.shape[1] for g in graphs) if graphs else 4

    print(f"  Training: {epochs} epochs, batch_size={batch_size}, lr={lr}")

    for epoch in range(epochs):
        epoch_losses = []
        optimizer.zero_grad()
        shuffled_graphs = graphs.copy()
        random.shuffle(shuffled_graphs)

        for batch_idx, i in enumerate(range(0, len(shuffled_graphs), batch_size)):
            batch_g = shuffled_graphs[i:i+batch_size]
            aug1 = [graph_augment(g, target_ch) for g in batch_g]
            aug2 = [graph_augment(g, target_ch) for g in batch_g]

            # Move to GPU
            for g in aug1 + aug2:
                g.x, g.edge_index = g.x.to(device), g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)

            try:
                b1, b2 = Batch.from_data_list(aug1), Batch.from_data_list(aug2)
            except:
                continue

            z1, p1 = model(b1.x, b1.edge_index, getattr(b1, 'edge_attr', None), b1.batch)
            z2, _ = model(b2.x, b2.edge_index, getattr(b2, 'edge_attr', None), b2.batch)

            # Contrastive loss
            loss_contrast = contrastive_loss(z1, z2)

            # Supervised losses on gt_score
            loss_reg = torch.tensor(0.0, device=device)
            loss_rank = torch.tensor(0.0, device=device)

            if hasattr(b1, 'gt_score'):
                try:
                    gt = b1.gt_score.to(device).float()
                    pred = p1.view(-1)

                    # 1. MSE Regression loss (normalized)
                    if gt.std() > 0:
                        gt_norm = (gt - gt.mean()) / (gt.std() + 1e-8)
                    else:
                        gt_norm = gt
                    if pred.std() > 0:
                        pred_norm = (pred - pred.mean()) / (pred.std() + 1e-8)
                    else:
                        pred_norm = pred
                    loss_reg = F.mse_loss(pred_norm, gt_norm)

                    # 2. Pairwise Ranking loss - learn correct ordering
                    # For pairs where gt[i] > gt[j], we want pred[i] > pred[j]
                    if len(gt) >= 2:
                        n_pairs = min(len(gt) * (len(gt) - 1) // 2, 100)  # Limit pairs
                        idx = torch.randperm(len(gt))[:min(20, len(gt))]  # Sample indices
                        loss_rank_sum = 0.0
                        n_valid = 0
                        for ii in range(len(idx)):
                            for jj in range(ii + 1, len(idx)):
                                i_idx, j_idx = idx[ii], idx[jj]
                                if gt[i_idx] > gt[j_idx]:
                                    # pred[i] should be > pred[j], margin = 0.1
                                    loss_rank_sum += F.relu(0.1 - (pred[i_idx] - pred[j_idx]))
                                    n_valid += 1
                                elif gt[i_idx] < gt[j_idx]:
                                    loss_rank_sum += F.relu(0.1 - (pred[j_idx] - pred[i_idx]))
                                    n_valid += 1
                        if n_valid > 0:
                            loss_rank = loss_rank_sum / n_valid
                except:
                    pass

            # Combined loss: contrastive + regression + ranking
            # Increase supervised weight for better ranking
            loss = (loss_contrast + 1.0 * loss_reg + 0.5 * loss_rank) / gradient_accumulation_steps
            loss.backward()

            epoch_losses.append(loss.item() * gradient_accumulation_steps)

            # Update weights every gradient_accumulation_steps
            if (batch_idx + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

            # Cleanup
            del z1, z2, b1, b2, aug1, aug2
            if device.type == 'cuda':
                torch.cuda.empty_cache()

        # Final update for remaining batches
        if len(shuffled_graphs) // batch_size % gradient_accumulation_steps != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = np.mean(epoch_losses) if epoch_losses else 0.0
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")

    return model

print("Model and training functions loaded!")

In [ ]:
# Cell 6: Coordinate Extraction & Accuracy (GPU Batch Processing)
# UPDATED: Ground truth based on INTERSECTION (voxels where 2+ channels are active)
# Comparison is coordinate-based, without considering rank and score

# Minimum number of channels that must be active for a voxel to be an intersection
MIN_INTERSECTION_CHANNELS = 2


def extract_intersection_coords(data):
    """Extract ALL intersection coordinates from ground truth data.

    An intersection is a voxel where MIN_INTERSECTION_CHANNELS or more channels are active.
    This finds ALL such voxels regardless of number (could be 10, 100, or 1000+).

    Args:
        data: numpy array of shape (num_channels, num_values, z_dim, y_dim, x_dim)

    Returns:
        List of dicts with 'x', 'y', 'z' coordinates of intersection voxels
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # channel_active[c, z, y, x] = True if channel c has any value > 0
    channel_active = (data > 0).any(axis=1)  # (C, Z, Y, X)

    # Count active channels per voxel
    channel_count = channel_active.sum(axis=0)  # (Z, Y, X)

    # Find voxels where 2+ channels are active (intersections)
    intersection_mask = channel_count >= MIN_INTERSECTION_CHANNELS

    # Get coordinates of intersection voxels
    intersection_coords = []
    z_indices, y_indices, x_indices = np.where(intersection_mask)

    for z, y, x in zip(z_indices, y_indices, x_indices):
        intersection_coords.append({
            'x': int(x),
            'y': int(y),
            'z': int(z)
        })

    print(f"    Found {len(intersection_coords)} intersection voxels (>={MIN_INTERSECTION_CHANNELS} channels)")
    return intersection_coords


def extract_agt_coords(data):
    """Extract ALL AGT (All Ground Truth) coordinates from ground truth data.

    AGT is a voxel where at least one channel has a value (at least 1 channel is active).
    This finds ALL such voxels regardless of number.

    Args:
        data: numpy array of shape (num_channels, num_values, z_dim, y_dim, x_dim)

    Returns:
        List of dicts with 'x', 'y', 'z' coordinates of AGT voxels
    """
    num_channels, num_values, z_dim, y_dim, x_dim = data.shape

    # channel_active[c, z, y, x] = True if channel c has any value > 0
    channel_active = (data > 0).any(axis=1)  # (C, Z, Y, X)

    # Count active channels per voxel
    channel_count = channel_active.sum(axis=0)  # (Z, Y, X)

    # Find voxels where at least 1 channel is active (AGT)
    agt_mask = channel_count >= 1

    # Get coordinates of AGT voxels
    agt_coords = []
    z_indices, y_indices, x_indices = np.where(agt_mask)

    for z, y, x in zip(z_indices, y_indices, x_indices):
        agt_coords.append({
            'x': int(x),
            'y': int(y),
            'z': int(z)
        })

    print(f"    Found {len(agt_coords)} AGT voxels (>=1 channel active)")
    return agt_coords


def find_model_predictions(model, graphs, device, batch_size=128):
    """Batch GPU inference to get ALL model predictions.

    Returns ALL graph centers with their scores (no top_k filtering).
    """
    model.eval()
    target_ch = max(g.x.shape[1] for g in graphs) if graphs else 4
    all_predictions = []

    with torch.no_grad():
        for i in range(0, len(graphs), batch_size):
            batch_g = graphs[i:i+batch_size]
            prepared = [prepare_graph(g, target_ch) for g in batch_g]

            for g in prepared:
                g.x, g.edge_index = g.x.to(device), g.edge_index.to(device)
                if hasattr(g, 'edge_attr') and g.edge_attr is not None:
                    g.edge_attr = g.edge_attr.to(device)

            try:
                batch = Batch.from_data_list(prepared)
                _, scores = model(batch.x, batch.edge_index, getattr(batch, 'edge_attr', None), batch.batch)
                scores = scores.cpu().numpy().flatten()

                for j, g in enumerate(batch_g):
                    all_predictions.append({
                        'x': int(g.center[0]),
                        'y': int(g.center[1]),
                        'z': int(g.center[2]),
                        'score': float(scores[j])
                    })
            except:
                for g in batch_g:
                    all_predictions.append({
                        'x': int(g.center[0]),
                        'y': int(g.center[1]),
                        'z': int(g.center[2]),
                        'score': 0.0
                    })

    print(f"    Model predictions: {len(all_predictions)} total")
    return all_predictions


def normalize_euclidean(distances):
    """Normalize Euclidean distances by dividing by EUCLIDEAN_STEP."""
    distances = np.array(distances)
    distances = distances / EUCLIDEAN_STEP
    return distances


def compute_distance_stats(model_coords, gt_coords):
    """Compute distance statistics between model predictions and GT coordinates.

    For each model prediction, finds the closest GT point and computes Euclidean distance
    using Z_SCALE_FACTOR for z-axis scaling.

    Args:
        model_coords: List of model predicted coordinates (dicts with 'x', 'y', 'z')
        gt_coords: List of ground truth coordinates (dicts with 'x', 'y', 'z')

    Returns:
        Dictionary with 'mean', 'median', 'std', 'min', 'max' for normalized distances
    """
    n_model = len(model_coords)
    n_gt = len(gt_coords)

    if n_model == 0:
        return {'mean': 0.0, 'median': 0.0, 'std': 0.0, 'min': 0.0, 'max': 0.0}

    if n_gt == 0:
        return {'mean': float('inf'), 'median': float('inf'), 'std': 0.0,
                'min': float('inf'), 'max': float('inf')}

    model_pts = np.array([[c['x'], c['y'], c['z']] for c in model_coords], dtype=np.float32)
    gt_pts = np.array([[c['x'], c['y'], c['z']] for c in gt_coords], dtype=np.float32)

    # Compute minimum distances from each model point to any GT point
    min_distances = []
    batch_size = 500  # Process model predictions in batches

    for i in range(0, n_model, batch_size):
        batch_model = model_pts[i:i+batch_size]

        # Compute distances from this batch to all GT points
        # Apply Z_SCALE_FACTOR to account for different z resolution
        diff = batch_model[:, np.newaxis, :] - gt_pts[np.newaxis, :, :]
        diff[:, :, 2] = diff[:, :, 2] * Z_SCALE_FACTOR  # Scale z dimension
        dist = np.sqrt(np.sum(diff**2, axis=2))  # (batch_size, n_gt)

        # For each model point, get min distance to any GT
        min_dist = dist.min(axis=1)
        min_distances.extend(min_dist.tolist())

        del diff, dist  # Free memory

    # Normalize distances
    if len(min_distances) > 0:
        normalized = normalize_euclidean(np.array(min_distances))

        # Calculate initial mean and std for outlier detection
        initial_mean = np.mean(normalized)
        initial_std = np.std(normalized)

        # Remove outliers: keep only values within mean ± 3*std
        if initial_std > 0:
            outlier_threshold = 3 * initial_std
            filtered_distances = normalized[
                (normalized >= initial_mean - outlier_threshold) &
                (normalized <= initial_mean + outlier_threshold)
            ]
        else:
            filtered_distances = normalized

        # Recalculate statistics on filtered data (more accurate)
        if len(filtered_distances) > 0:
            mean = float(np.mean(filtered_distances))
            median = float(np.median(filtered_distances))
            std = float(np.std(filtered_distances))
            min_val = float(np.min(filtered_distances))
            max_val = float(np.max(filtered_distances))
        else:
            # Fallback to original if all filtered out
            mean = float(initial_mean)
            median = float(np.median(normalized))
            std = float(initial_std)
            min_val = float(np.min(normalized))
            max_val = float(np.max(normalized))
    else:
        mean = median = std = min_val = max_val = 0.0

    return {'mean': mean, 'median': median, 'std': std, 'min': min_val, 'max': max_val}


def compute_accuracy(model_coords, gt_intersection_coords, tol=None, outlier_percentile=10):
    """Compute accuracy: for each model prediction, find closest GT and compute Euclidean distance.

    Ground truth is based on intersection (voxels with 2+ channels active).
    For each model prediction, finds the closest GT intersection and computes Euclidean distance
    using Z_SCALE_FACTOR for z-axis scaling (z resolution is half of x,y resolution).

    Args:
        model_coords: List of model predicted coordinates
        gt_intersection_coords: List of ground truth intersection coordinates (2+ channels overlap)
        tol: Match tolerance in voxels (default: MATCH_TOLERANCE)

    Returns:
        Dictionary with TP, FP, FN, precision, recall, f1_score, accuracy, and distance stats
        (mean, median, std, min, max) for ALL model predictions to their closest GT
    """
    if tol is None:
        tol = MATCH_TOLERANCE

    n_model = len(model_coords)
    n_gt = len(gt_intersection_coords)

    if n_model == 0:
        return {'TP': 0, 'FP': 0, 'FN': n_gt, 'precision': 0, 'recall': 0,
                'f1_score': 0, 'accuracy_iou': 0, 'model_count': 0, 'gt_count': n_gt,
                'euclidean_mean': 0, 'euclidean_median': 0, 'euclidean_std': 0, 'euclidean_min': 0, 'euclidean_max': 0}

    if n_gt == 0:
        return {'TP': 0, 'FP': n_model, 'FN': 0, 'precision': 0, 'recall': 0,
                'f1_score': 0, 'accuracy_iou': 0, 'model_count': n_model, 'gt_count': 0,
                'euclidean_mean': float('inf'), 'euclidean_median': float('inf'), 'euclidean_std': 0,
                'euclidean_min': float('inf'), 'euclidean_max': float('inf')}

    model_pts = np.array([[c['x'], c['y'], c['z']] for c in model_coords], dtype=np.float32)
    gt_pts = np.array([[c['x'], c['y'], c['z']] for c in gt_intersection_coords], dtype=np.float32)

    # For each model prediction, check if ANY GT is within tolerance
    # Use batched computation for memory efficiency
    TP = 0
    batch_size = 500  # Process model predictions in batches
    min_distances = []

    for i in range(0, n_model, batch_size):
        batch_model = model_pts[i:i+batch_size]

        # Compute distances from this batch to all GT points
        # Apply Z_SCALE_FACTOR to account for different z resolution
        diff = batch_model[:, np.newaxis, :] - gt_pts[np.newaxis, :, :]
        diff[:, :, 2] = diff[:, :, 2] * Z_SCALE_FACTOR  # Scale z dimension
        dist = np.sqrt(np.sum(diff**2, axis=2))  # (batch_size, n_gt)

        # For each model point, check if min distance to any GT <= tolerance
        min_dist = dist.min(axis=1)
        min_distances.extend(min_dist.tolist())
        TP += int((min_dist <= tol).sum())

        del diff, dist  # Free memory

    # FP = model predictions with no GT nearby
    FP = n_model - TP

    # FN = GT points not matched by any model prediction
    # (compute which GT points are within tolerance of ANY model prediction)
    matched_gt = np.zeros(n_gt, dtype=bool)
    for i in range(0, n_model, batch_size):
        batch_model = model_pts[i:i+batch_size]
        diff = batch_model[:, np.newaxis, :] - gt_pts[np.newaxis, :, :]
        diff[:, :, 2] = diff[:, :, 2] * Z_SCALE_FACTOR  # Scale z dimension
        dist = np.sqrt(np.sum(diff**2, axis=2))

        # Mark GT points that are matched by this batch
        matched_gt |= (dist.min(axis=0) <= tol)
        del diff, dist

    FN = int((~matched_gt).sum())

    # Metrics
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    iou = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0

    # Distance statistics for ALL model predictions to their closest GT
    # Compute Euclidean distance for all predictions (not just non-matching ones)
    # Remove outliers: values beyond mean ± 3*std
    if len(min_distances) > 0:
        normalized = normalize_euclidean(np.array(min_distances))

        # Calculate initial mean and std for outlier detection
        initial_mean = np.mean(normalized)
        initial_std = np.std(normalized)

        # Remove outliers: keep only values within mean ± 3*std
        if initial_std > 0:
            outlier_threshold = 3 * initial_std
            filtered_distances = normalized[
                (normalized >= initial_mean - outlier_threshold) &
                (normalized <= initial_mean + outlier_threshold)
            ]
        else:
            filtered_distances = normalized

        # Recalculate statistics on filtered data (more accurate)
        if len(filtered_distances) > 0:
            euclidean_mean = float(np.mean(filtered_distances))
            euclidean_median = float(np.median(filtered_distances))
            euclidean_std = float(np.std(filtered_distances))
            euclidean_min = float(np.min(filtered_distances))
            euclidean_max = float(np.max(filtered_distances))
        else:
            # Fallback to original if all filtered out
            euclidean_mean = float(initial_mean)
            euclidean_median = float(np.median(normalized))
            euclidean_std = float(initial_std)
            euclidean_min = float(np.min(normalized))
            euclidean_max = float(np.max(normalized))
    else:
        euclidean_mean = euclidean_median = euclidean_std = euclidean_min = euclidean_max = 0.0

    return {'TP': TP, 'FP': FP, 'FN': FN, 'precision': precision, 'recall': recall,
            'f1_score': f1, 'accuracy_iou': iou, 'model_count': n_model, 'gt_count': n_gt,
            'tolerance': tol, 'euclidean_mean': euclidean_mean, 'euclidean_median': euclidean_median,
            'euclidean_std': euclidean_std, 'euclidean_min': euclidean_min, 'euclidean_max': euclidean_max}


print("Coordinate extraction and accuracy functions loaded!")
print(f"Intersection threshold: {MIN_INTERSECTION_CHANNELS}+ channels active")
print(f"Z Scale Factor: {Z_SCALE_FACTOR} (z differences multiplied by this in Euclidean distance)")

In [ ]:
# Cell 7: Main Execution - Run All Experiments
# UPDATED: Compares ALL model predictions with ALL GT intersections (no top_k)

def print_table(texture_name, results):
    """Print formatted table for one texture with intersection-based comparison"""
    print(f"\n{'='*140}")
    print(f"RESULTS TABLE: {texture_name.upper()} (All GT vs All Model Predictions)")
    print(f"{'='*140}")
    print(f"{'Scale Name':>12} {'Scale':>12} {'Model':>8} {'GT':>8} {'TP':>8} {'FP':>8} {'FN':>8} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    print("-"*140)
    for r in results:
        print(f"{r['Scale Name']:>12} {r['Scale']:>12} {r['Model Points']:>8} {r['GT Points']:>8} {r['TP']:>8} {r['FP']:>8} {r['FN']:>8} {r['Precision']:>10.4f} {r['Recall']:>8.4f} {r['F1 Score']:>8.4f}")
    print("="*140)

    # Print distance statistics table
    print(f"\n{'='*180}")
    print(f"DISTANCE STATISTICS: {texture_name.upper()} (Euclidean distance from each model prediction to closest GT)")
    print(f"{'='*180}")
    print(f"{'Scale Name':>12} {'Scale':>12} {'GT Mean':>10} {'GT Median':>10} {'GT Std':>10} {'AGT Mean':>10} {'AGT Median':>10} {'AGT Std':>10}")
    print("-"*180)
    for r in results:
        print(f"{r['Scale Name']:>12} {r['Scale']:>12} {r['Euc Mean']:>10.4f} {r['Euc Median']:>10.4f} {r['Euc Std']:>10.4f} {r['AGT Mean']:>10.4f} {r['AGT Median']:>10.4f} {r['AGT Std']:>10.4f}")
    print("="*180)


def run_texture(texture_type, device):
    """Run all scales for one texture"""
    print(f"\n{'*'*80}")
    print(f"TEXTURE: {texture_type.upper()}")
    print(f"{'*'*80}")

    texture_dir = os.path.join(BASE_DIR, texture_type)
    os.makedirs(texture_dir, exist_ok=True)

    results = []

    for scale_name, x_s, y_s, z_s in tqdm(SCALES, desc=f"{texture_type}"):
        try:
            # 1. Create ground truth
            data, gt_path = create_groundtruth(texture_type, scale_name, x_s, y_s, z_s, texture_dir)

            # 2. Create subgraphs
            graphs = create_subgraphs(data, texture_type, scale_name, texture_dir)

            # 3. Filter graphs (use reasonable min_nodes based on K)
            MIN_NODES = min(10, 100 // (K * K))
            filtered = [g for g in graphs if g.x.shape[0] >= MIN_NODES]

            if not filtered:
                print(f"  WARNING {scale_name}: No graphs passed filter (total: {len(graphs)}, min_nodes={MIN_NODES})")
                MIN_NODES = 2
                filtered = [g for g in graphs if g.x.shape[0] >= MIN_NODES]
                if not filtered:
                    continue

            print(f"  {scale_name}: {len(filtered)} graphs after filter (min_nodes={MIN_NODES})")

            # 4. Train model (GPU) - No GT scores needed, all intersections are equal
            # 5. Train model (GPU)
            in_ch = max(g.x.shape[1] for g in filtered)
            edge_dim = 1 if any(hasattr(g, 'edge_attr') and g.edge_attr is not None for g in filtered) else None
            model = ContrastiveGAT(in_ch, 32, 16, 4, 0.1, edge_dim).to(device)

            train_graphs = filtered[::2] if len(filtered) > 20000 else filtered
            model = train_model_gpu(model, train_graphs, device, epochs=1, batch_size=32, lr=0.01, gradient_accumulation_steps=4)

            # 6. Extract ALL model predictions (no top_k filtering)
            model_coords = find_model_predictions(model, filtered, device, batch_size=128)

            # 7. Extract ALL GT intersection coordinates
            # GT is based on intersections (voxels with 2+ channels active)
            gt_intersection_coords = extract_intersection_coords(data)

            # 7b. Extract ALL AGT coordinates
            # AGT is based on all voxels with at least 1 channel active
            gt_agt_coords = extract_agt_coords(data)

            # Save intersection coordinates to file
            intersection_file = os.path.join(texture_dir, f'gt_intersections_{texture_type}_{scale_name}.json')
            import json
            with open(intersection_file, 'w') as f:
                json.dump({
                    'texture': texture_type,
                    'scale': scale_name,
                    'min_channels': MIN_INTERSECTION_CHANNELS,
                    'count': len(gt_intersection_coords),
                    'coordinates': gt_intersection_coords
                }, f, indent=2)

            # 8. Compute accuracy: compare ALL model coords with ALL GT intersection coords
            # For each model prediction, find closest GT intersection and compute Euclidean distance
            # Ground truth is based on intersection (voxels with 2+ channels active)
            acc = compute_accuracy(model_coords, gt_intersection_coords, MATCH_TOLERANCE)

            # 8b. Compute distance statistics for AGT
            agt_stats = compute_distance_stats(model_coords, gt_agt_coords)

            results.append({
                'Scale Name': scale_name,
                'Scale': f'{x_s}x,{y_s}x,{z_s}x',
                'TP': acc['TP'],
                'FP': acc['FP'],
                'FN': acc['FN'],
                'Model Points': acc['model_count'],
                'GT Points': acc['gt_count'],
                'Precision': acc['precision'],
                'Recall': acc['recall'],
                'F1 Score': acc['f1_score'],
                'Accuracy': acc['accuracy_iou'],
                'Tolerance': MATCH_TOLERANCE,
                # Distance statistics for ALL model predictions to their closest GT intersection
                'Euc Mean': acc['euclidean_mean'],
                'Euc Median': acc['euclidean_median'],
                'Euc Std': acc['euclidean_std'],
                'Euc Min': acc['euclidean_min'],
                'Euc Max': acc['euclidean_max'],
                # Distance statistics for AGT (All Ground Truth - at least 1 channel active)
                'AGT Mean': agt_stats['mean'],
                'AGT Median': agt_stats['median'],
                'AGT Std': agt_stats['std'],
                'AGT Min': agt_stats['min'],
                'AGT Max': agt_stats['max']
            })

            # Cleanup
            del data, graphs, filtered, model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"  ERROR {scale_name}: {e}")
            import traceback
            traceback.print_exc()
            continue

    # Print table
    if results:
        print_table(texture_type, results)
        df = pd.DataFrame(results)
        csv_path = os.path.join(texture_dir, f'results_{texture_type}.csv')
        df.to_csv(csv_path, index=False)
        print(f"  Saved: {csv_path}")
    else:
        print(f"  WARNING: No results for {texture_type}")

    return results

print("Execution functions loaded!")

In [ ]:
# Cell 8: RUN ALL EXPERIMENTS

print("="*80)
print("MULTI-TEXTURE SCALE EXPERIMENT - GPU VERSION")
print("="*80)
print(f"\nDevice: {device}")
print(f"Textures: {TEXTURES}")
print(f"Total experiments: {len(TEXTURES) * len(SCALES)}")

start_time = time.time()
all_results = {}

# Run each texture
for texture in TEXTURES:
    all_results[texture] = run_texture(texture, device)

total_time = time.time() - start_time
print(f"\nTotal time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")

Oval									

Oval									
Scale Name	Scale	Model	GT	TP	FP	FN	Precision	Recall	F1
original	1x,1x,1x	2178	2173	1847	331	326	0.848	0.85	0.849
2x1y1z	2x,1x,1x	2177	2823	1960	217	863	0.9003	0.6943	0.784
1x2y1z	1x,2x,1x	5246	5754	4587	659	1167	0.8744	0.7972	0.834
1x1y2z	1x,1x,2x	2178	2297	1969	209	328	0.904	0.8572	0.88
2x2y1z	2x,2x,1x	8239	8961	7353	886	1608	0.8925	0.8206	0.855
1x2y2z	1x,2x,2x	5245	4958	4423	822	535	0.8433	0.8921	0.867
2x2y2z	2x,2x,2x	8239	8761	7123	1116	1638	0.8645	0.813	0.838
3x1y1z	3x,1x,1x	2178	3435	1900	278	1535	0.8724	0.5531	0.677
3x2y1z	3x,2x,1x	8250	13750	7403	847	6347	0.8973	0.5384	0.673
3x1y2z	3x,1x,2x	2178	3822	1878	300	1944	0.8623	0.4914	0.626
3x2y2z	3x,2x,2x	8250	14750	7452	798	7298	0.9033	0.5052	0.648


Olympic Ring

Olympic-Ring					
Scale Name	Scale	Model	GT	TP	FP	FN	Precision	Recall	F1	
original	1x,1x,1x	673	623	600	73	23	0.891530461	0.963081862	0.9321	
2x1y1z	2x,1x,1x	785	709	655	130	54	0.8344	0.923836389	0.8768	
1x2y1z	1x,2x,1x	2169	2226	1971	198	255	0.9087	0.8854	0.8969	
1x1y2z	1x,1x,2x	708	707	643	65	64	0.9082	0.9095	0.9088	
2x2y1z	2x,2x,1x	2317	2486	1931	386	555	0.8334	0.7767	0.8041	
1x2y2z	1x,2x,2x	2169	2580	1971	198	609	0.9087	0.764	0.8301	
2x2y2z	2x,2x,2x	2317	2826	2106	211	720	0.9089	0.7452	0.819	
3x1y1z	3x,1x,1x	791	1162	660	131	502	0.8344	0.568	0.6759	
3x2y1z	3x,2x,1x	2520	2949	2100	420	849	0.8333	0.7121	0.768	
3x1y2z	3x,1x,2x	791	993	660	131	333	0.8344	0.6647	0.7399	
3x2y2z	3x,2x,2x	2522	3051	2102	420	949	0.8333	0.6885	0.754	


Linear

	Linear									
Scale Name	Scale	Model	GT	TP	FP	FN	Precision	Recall	F1	
original	1x,1x,1x	1156	1156	1099	57	57	0.951	0.951	0.951	
2x1y1z	2x,1x,1x	2056	2056	1721	335	335	0.837	0.837	0.837	
1x2y1z	1x,2x,1x	2517	2517	2086	431	431	0.829	0.829	0.829	
1x1y2z	1x,1x,2x	1156	1156	973	183	183	0.841	0.841	0.841	
2x2y1z	2x,2x,1x	3879	3879	3157	722	722	0.814	0.814	0.814	
1x2y2z	1x,2x,2x	2517	2517	2103	414	414	0.836	0.836	0.836	
2x2y2z	2x,2x,2x	3879	3879	3231	648	648	0.833	0.833	0.833	
3x1y1z	3x,1x,1x	3208	4278	2246	962	2032	0.7	0.525	0.6	
3x2y1z	3x,2x,1x	6563	6563	4660	1903	1903	0.71	0.71	0.71	
3x1y2z	3x,1x,2x	3208	3208	2271	937	937	0.708	0.708	0.708	
3x2y2z	3x,2x,2x	6563	6563	4717	1846	1846	0.719	0.719	0.719	


Colonies

Colonies									
Scale Name	Scale	Model	GT	TP	FP	FN	Precision	Recall	F1	
original	1x,1x,1x	55	56	49	6	7	0.890909091	0.875	0.882882883	
2x1y1z	2x,1x,1x	107	111	92	15	19	0.859813084	0.828828829	0.844036697	
1x2y1z	1x,2x,1x	83	85	71	12	14	0.855421687	0.835294118	0.845238095	
1x1y2z	1x,1x,2x	69	72	60	9	12	0.869565217	0.833333333	0.85106383	
2x2y1z	2x,2x,1x	143	149	117	26	32	0.818181818	0.785234899	0.801369863	
1x2y2z	1x,2x,2x	88	92	74	14	18	0.840909091	0.804347826	0.822222222	
2x2y2z	2x,2x,2x	140	146	118	22	28	0.842857143	0.808219178	0.825174825	
3x1y1z	3x,1x,1x	167	175	115	52	60	0.688622754	0.657142857	0.67251462	
3x2y1z	3x,2x,1x	224	234	169	55	65	0.754464286	0.722222222	0.737991266	
3x1y2z	3x,1x,2x	119	125	85	34	40	0.714285714	0.68	0.696721311	
3x2y2z	3x,2x,2x	192	202	147	45	55	0.765625	0.727722772	0.746192893	


Sinusuide

Sinusuide									
Scale Name	Scale	Model	GT	TP	FP	FN	Precision	Recall	F1
original	1x,1x,1x	1307	1339	1199	108	140	0.9127	0.8898	0.911
2x1y1z	2x,1x,1x	2364	2420	2125	239	295	0.899	0.8781	0.899
1x2y1z	1x,2x,1x	2332	2397	1905	427	492	0.8102	0.7875	0.81
1x1y2z	1x,1x,2x	1368	1425	1196	172	229	0.8611	0.8228	0.861
2x2y1z	2x,2x,1x	3922	4035	3281	641	754	0.8473	0.8249	0.824
1x2y2z	1x,2x,2x	2397	2467	1962	435	505	0.8067	0.7828	0.807
2x2y2z	2x,2x,2x	4414	4532	3527	887	1005	0.7886	0.7672	0.789
3x1y1z	3x,1x,1x	3802	3879	3355	447	524	0.8731	0.8544	0.874
3x2y1z	3x,2x,1x	6434	6559	5118	1316	1441	0.7869	0.7713	0.787
3x1y2z	3x,1x,2x	3694	3786	2965	729	821	0.7929	0.7729	0.793
3x2y2z	3x,2x,2x	6475	6650	4979	1496	1671	0.758	0.737	0.758
